# Lab 5: Hyperspectral Data & Water Quality Analysis

## Introduction
The aim of this project is to develop tools enabling the fusion of airborne data (hyperspectral) and satellite data (multispectral) in order to build water quality monitoring tools.

**Tasks to complete:**
1. Develop a Python tool for browsing the data cube (RGB visualization, spectral signature extraction, CSV export).
2. Prepare spectral signatures for different land covers (water, green areas, forest) to create a spectral library.
3. Calculate and compare water quality indices (Chl-a, DOC, turbidity) between Airborne and Sentinel-2 data.
4. Prepare SAM and calibrate Sentinel-2 data using Airborne data.

In [ ]:
# Install missing libraries if needed
# %pip install spectral rasterio scikit-learn matplotlib numpy

import os
import sys
import csv
import tkinter as tk
from tkinter import filedialog, messagebox
from pathlib import Path

import numpy as np
import rasterio
from matplotlib import pyplot as plt
import matplotlib
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.linear_model import LinearRegression
import spectral.io.envi as envi

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## Task 1 & 2: Hyperspectral Data Viewer & Spectral Library Generation

The cell below contains the interactive GUI viewer. 
**Important:** When you run this cell, an external window will pop up on your computer. 
1. Use it to open your `.hdr` file.
2. Click on different pixels (Water, Forest, Bare Soil).
3. Export the signatures to `.csv` files (This fulfills **Task 2**).
4. **Close the window** to continue running the rest of this notebook.

In [ ]:
# Default search directory 
DATA_DIR = Path("data/images")
FALLBACK_RGB = (30, 20, 10)

def find_hdr_files(directory: Path) -> list[Path]:
    return sorted(directory.glob("*.hdr"))

def parse_wavelengths(meta: dict) -> np.ndarray | None:
    wl = meta.get("wavelength")
    if wl: return np.array([float(w) for w in wl])
    return None

def get_rgb_bands(meta: dict) -> tuple[int, int, int]:
    db = meta.get("default bands")
    if db and len(db) >= 3: return tuple(int(float(v)) - 1 for v in db[:3])
    return FALLBACK_RGB

def get_ignore_value(meta: dict) -> float | None:
    raw = meta.get("data ignore value")
    if raw:
        try: return float(str(raw).strip())
        except ValueError: pass
    return None

def load_image(hdr_path: Path):
    return envi.open(str(hdr_path))

def read_rgb(img, r: int, g: int, b: int, ignore_value: float | None) -> np.ndarray:
    rgb = img.read_bands([r, g, b]).astype(np.float32)
    if ignore_value is not None: rgb[rgb >= ignore_value] = np.nan
    rgb[rgb < 0] = np.nan
    for c in range(3):
        ch = rgb[:, :, c]
        p2, p98 = np.nanpercentile(ch, [2, 98])
        rgb[:, :, c] = np.clip((ch - p2) / max(p98 - p2, 1e-6), 0, 1)
    return np.nan_to_num(rgb, nan=0.0)

def read_spectrum(img, row: int, col: int, ignore_value: float | None) -> np.ndarray:
    spec = img.read_pixel(row, col).astype(np.float64)
    if ignore_value is not None: spec[spec >= ignore_value] = np.nan
    spec[spec < 0] = np.nan
    return spec

class HyperspectralViewer:
    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("Hyperspectral BSQ Viewer")
        self.root.geometry("1300x720")
        self.img, self.wavelengths, self.ignore_value = None, None, None
        self.rgb_display, self.spectrum, self.pixel_pos = None, None, None
        self._build_ui()
        self._open_file() # Open dialog immediately for Jupyter workflow

    def _build_ui(self):
        bar = tk.Frame(self.root, bd=1, relief=tk.RAISED)
        bar.pack(side=tk.TOP, fill=tk.X)
        tk.Button(bar, text="Open file…", command=self._open_file).pack(side=tk.LEFT, padx=4, pady=3)
        tk.Button(bar, text="Export spectrum to CSV…", command=self._export_csv).pack(side=tk.LEFT, padx=4, pady=3)
        self.status_var = tk.StringVar(value="No file loaded.")
        tk.Label(bar, textvariable=self.status_var, anchor=tk.W, fg="#444").pack(side=tk.LEFT, padx=12)

        self.fig = Figure(figsize=(14, 6.5))
        self.ax_rgb = self.fig.add_subplot(1, 2, 1)
        self.ax_spec = self.fig.add_subplot(1, 2, 2)
        self.fig.tight_layout(pad=2.5)

        self.canvas = FigureCanvasTkAgg(self.fig, master=self.root)
        NavigationToolbar2Tk(self.canvas, self.root)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
        self.canvas.mpl_connect("button_press_event", self._on_click)

    def _open_file(self):
        path = filedialog.askopenfilename(
            title="Open ENVI header (.hdr)",
            initialdir=DATA_DIR if DATA_DIR.exists() else Path.home(),
            filetypes=[("ENVI header", "*.hdr"), ("All files", "*.*")],
        )
        if path: self._load(Path(path))

    def _load(self, hdr_path: Path):
        self.status_var.set(f"Loading RGB bands from  {hdr_path.name} …")
        self.root.update_idletasks()
        try:
            self.img = load_image(hdr_path)
            meta = self.img.metadata
            self.wavelengths = parse_wavelengths(meta)
            self.ignore_value = get_ignore_value(meta)
            r, g, b = get_rgb_bands(meta)
            self.rgb_display = read_rgb(self.img, r, g, b, self.ignore_value)
            self.spectrum, self.pixel_pos = None, None
            self._refresh_plots()
            self.status_var.set(f"{hdr_path.name} loaded. Click a pixel to inspect its spectrum.")
        except Exception as exc:
            messagebox.showerror("Error", str(exc))
            self.status_var.set("Load failed.")

    def _refresh_plots(self):
        self.ax_rgb.clear()
        if self.rgb_display is not None:
            self.ax_rgb.imshow(self.rgb_display, interpolation="bilinear", aspect="auto")
            if self.pixel_pos:
                self.ax_rgb.plot(self.pixel_pos[1], self.pixel_pos[0], "r+", markersize=14, markeredgewidth=2.5)
        self.ax_rgb.set_title("RGB preview — click a pixel")
        self.ax_rgb.axis("off")

        self.ax_spec.clear()
        if self.spectrum is not None:
            x = self.wavelengths if self.wavelengths is not None else np.arange(len(self.spectrum))
            xlabel = "Wavelength (nm)" if self.wavelengths is not None else "Band index"
            self.ax_spec.plot(x, self.spectrum, linewidth=1.2, color="steelblue")
            self.ax_spec.set_title(f"Spectral signature — row {self.pixel_pos[0]},  col {self.pixel_pos[1]}")
            self.ax_spec.set_xlabel(xlabel)
            self.ax_spec.set_ylabel("Reflectance")
            self.ax_spec.grid(True, alpha=0.3)
        self.fig.tight_layout(pad=2.5)
        self.canvas.draw()

    def _on_click(self, event):
        if event.inaxes is not self.ax_rgb or self.img is None: return
        col, row = int(round(event.xdata)), int(round(event.ydata))
        if not (0 <= row < self.img.nrows and 0 <= col < self.img.ncols): return
        self.pixel_pos = (row, col)
        self.spectrum = read_spectrum(self.img, row, col, self.ignore_value)
        self._refresh_plots()

    def _export_csv(self):
        if self.spectrum is None: return
        path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV", "*.csv")])
        if not path: return
        x = self.wavelengths if self.wavelengths is not None else np.arange(len(self.spectrum))
        header = "wavelength_nm" if self.wavelengths is not None else "band"
        with open(path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([header, "value"])
            for xi, vi in zip(x, self.spectrum): writer.writerow([float(xi), "" if np.isnan(vi) else float(vi)])
        messagebox.showinfo("Saved", f"Spectrum exported to:\n{path}")

# Run the GUI
matplotlib.use("TkAgg")
root = tk.Tk()
app = HyperspectralViewer(root)
root.mainloop()
# Reset matplotlib backend for the rest of the notebook inline plots
matplotlib.use("module://matplotlib_inline.backend_inline")

## Task 3: False-Color Composites & Water Quality Indices

Now we calculate standard water quality indices using both Airborne Data and Sentinel-2 data:
* **Turbidity (NDTI):** (Red - Green) / (Red + Green)
* **Chlorophyll-a (NDCI):** (NIR - Red) / (NIR + Red)
* **DOC / CDOM:** Blue / Green

*Note: Please update the file paths and band indices based on your specific datasets.*

In [ ]:
def calculate_indices(blue, green, red, nir):
    """Helper function to calculate water quality indices"""
    # Create mask to avoid division by zero
    valid_mask = (blue > 0) & (green > 0) & (red > 0) & (nir > 0)
    
    turbidity = np.full_like(red, np.nan)
    chl_a = np.full_like(red, np.nan)
    doc = np.full_like(red, np.nan)
    
    # 1. Turbidity
    turbidity[valid_mask] = (red[valid_mask] - green[valid_mask]) / (red[valid_mask] + green[valid_mask])
    # 2. Chlorophyll-a
    chl_a[valid_mask] = (nir[valid_mask] - red[valid_mask]) / (nir[valid_mask] + red[valid_mask])
    # 3. DOC
    doc[valid_mask] = blue[valid_mask] / green[valid_mask]
    
    return turbidity, chl_a, doc

def plot_indices(turbidity, chl_a, doc, title_prefix):
    """Helper function to plot the indices"""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    im1 = axes[0].imshow(turbidity, cmap='YlOrBr', vmin=-0.2, vmax=0.2)
    axes[0].set_title(f'{title_prefix} Turbidity')
    plt.colorbar(im1, ax=axes[0])

    im2 = axes[1].imshow(chl_a, cmap='YlGn', vmin=-0.2, vmax=0.5)
    axes[1].set_title(f'{title_prefix} Chlorophyll-a')
    plt.colorbar(im2, ax=axes[1])

    im3 = axes[2].imshow(doc, cmap='coolwarm', vmin=0.5, vmax=1.5)
    axes[2].set_title(f'{title_prefix} DOC')
    plt.colorbar(im3, ax=axes[2])

    plt.tight_layout()
    plt.show()

# ==========================================
# 1. AIRBORNE DATA PROCESSING
# ==========================================
# TODO: Update with your Airborne data path and matching band indices
hs_path = 'data/images/your_hyperspectral_image.bsq' 

try:
    with rasterio.open(hs_path) as src:
        # Example indices: match these to your .hdr wavelengths!
        airborne_blue = src.read(10).astype('float32')
        airborne_green = src.read(20).astype('float32')
        airborne_red = src.read(30).astype('float32')
        airborne_nir = src.read(40).astype('float32')

        # False Color Composite (NIR, Red, Green)
        vmax = np.nanpercentile(airborne_nir, 99)
        cir_composite = np.dstack((airborne_nir/vmax, airborne_red/vmax, airborne_green/vmax))
        cir_composite = np.clip(cir_composite, 0, 1)

        plt.figure(figsize=(6, 6))
        plt.imshow(cir_composite)
        plt.title('Airborne False-Color Composite')
        plt.axis('off')
        plt.show()

        # Calculate and plot Airborne Indices
        a_turb, a_chl, a_doc = calculate_indices(airborne_blue, airborne_green, airborne_red, airborne_nir)
        plot_indices(a_turb, a_chl, a_doc, "Airborne")

except Exception as e:
    print(f"Airborne Processing skipped/failed: {e}")

# ==========================================
# 2. SENTINEL-2 DATA PROCESSING
# ==========================================
# TODO: Download S2 imagery and update paths
s2_paths = {
    'blue': 'data/sentinel2/B02.jp2',
    'green': 'data/sentinel2/B03.jp2',
    'red': 'data/sentinel2/B04.jp2',
    'nir': 'data/sentinel2/B08.jp2'
}

try:
    with rasterio.open(s2_paths['blue']) as src: s2_blue = src.read(1).astype('float32')
    with rasterio.open(s2_paths['green']) as src: s2_green = src.read(1).astype('float32')
    with rasterio.open(s2_paths['red']) as src: s2_red = src.read(1).astype('float32')
    with rasterio.open(s2_paths['nir']) as src: s2_nir = src.read(1).astype('float32')
    
    s2_turb, s2_chl, s2_doc = calculate_indices(s2_blue, s2_green, s2_red, s2_nir)
    plot_indices(s2_turb, s2_chl, s2_doc, "Sentinel-2")

except Exception as e:
    print(f"Sentinel-2 Processing skipped/failed: {e}\nPlease download S2 data and update the paths.")

## Task 4: SAM & Sentinel-2 Calibration

**Spectral Angle Mapper (SAM):** Identifies materials by calculating the angle between a pixel's spectrum and a reference spectrum.

**Calibration Approach:** Airborne sensors are typically highly calibrated and atmospherically corrected. We can calibrate Sentinel-2 using Empirical Line Calibration (Linear Regression). We map overlapping Sentinel-2 pixels to Airborne pixels to find the equation: `Airborne = a * Sentinel + b`.

In [ ]:
def calculate_sam(image_spectra, reference_spectrum):
    """Calculates SAM between a 2D image array (pixels, bands) and a 1D reference spectrum."""
    dot_product = np.dot(image_spectra, reference_spectrum)
    norm_image = np.linalg.norm(image_spectra, axis=1)
    norm_ref = np.linalg.norm(reference_spectrum)
    
    # Clip to avoid arccos NaN issues
    cosine_theta = np.clip(np.divide(dot_product, (norm_image * norm_ref)), -1.0, 1.0)
    return np.arccos(cosine_theta)

def calibrate_sentinel2_band(s2_band_data, airborne_band_data):
    """Trains a linear regression model to map S2 data to Airborne data."""
    valid_mask = ~np.isnan(s2_band_data) & ~np.isnan(airborne_band_data)
    X = s2_band_data[valid_mask].reshape(-1, 1) # Feature
    y = airborne_band_data[valid_mask].reshape(-1, 1) # Target
    
    model = LinearRegression()
    model.fit(X, y)
    print(f"Calibration Eq: Airborne = {model.coef_[0][0]:.4f} * Sentinel2 + {model.intercept_[0]:.4f} (R2: {model.score(X, y):.4f})")
    return model

# ==========================================
# SAM EXAMPLE TEST
# ==========================================
# Creating dummy data for demonstration (replace with your actual image data)
cols, rows, bands = 100, 100, 6
dummy_image = np.random.rand(bands, rows, cols)
spectra_2d = dummy_image.reshape(bands, cols * rows).T

# Create a mock reference spectrum (In reality, load from the CSV generated in Task 2)
mock_ref_spectrum = np.array([0.1, 0.15, 0.08, 0.4, 0.2, 0.1])
sam_result_1d = calculate_sam(spectra_2d, mock_ref_spectrum)
sam_map = sam_result_1d.reshape(rows, cols)

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_title('Simulated SAM Map (Lower values = better match)')
img_plot = ax.imshow(sam_map, cmap='jet_r')
plt.colorbar(img_plot, fraction=0.046, pad=0.04)
plt.show()

# ==========================================
# CALIBRATION EXAMPLE TEST
# ==========================================
print("--- Band Calibration Simulation ---")
np.random.seed(42)
# Simulate 5000 overlapping pixels
sim_s2_red = np.random.uniform(0.05, 0.3, 5000)
sim_airborne_red = (sim_s2_red * 1.05) - 0.02 + np.random.normal(0, 0.01, 5000) 

# Train model
red_model = calibrate_sentinel2_band(sim_s2_red, sim_airborne_red)

# Apply to a full S2 image simulation
full_s2_image = np.random.uniform(0.05, 0.3, (300, 300))
calibrated_image = red_model.predict(full_s2_image.flatten().reshape(-1, 1)).reshape(300, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(full_s2_image, cmap='gray', vmin=0, vmax=0.4)
ax1.set_title("Original Sentinel-2")
ax2.imshow(calibrated_image, cmap='gray', vmin=0, vmax=0.4)
ax2.set_title("Calibrated Sentinel-2")
plt.show()